# contextlib: Setup and Cleanup Helpers

`contextlib` gives you tools for code that must run **setup** before and **cleanup** after a block, even when an error happens in the middle. The `with` statement is the usual way to use them.

## 1. What `with` does

`with` runs setup, runs your block, then runs cleanup, even if the block raises an error. Files are the most common example:

In [ ]:
from pathlib import Path
import tempfile

path = Path(tempfile.mkdtemp()) / "demo.txt"

with open(path, "w") as f:
    f.write("hello")

print("file closed after the block:", f.closed)

After the block, `f.closed` is `True`, even though we never called `f.close()`. The object that `with` manages is called a **context manager**. Any object with `__enter__` and `__exit__` methods works.

## 2. `closing()`: call `close()` for you

Some objects have a `close()` method but aren't context managers. `contextlib.closing` wraps them so `close()` runs at the end of the block:

In [ ]:
from contextlib import closing
import sqlite3

with closing(sqlite3.connect(":memory:")) as conn:
    print(conn.execute("select 1 + 1").fetchone())

try:
    conn.execute("select 1")
except sqlite3.ProgrammingError as err:
    print("connection is closed:", err)

## 3. Why `sqlite3.connect` needs `closing()`

This surprises many people. Using a connection in a `with` block does **not** close it. The `with` block only commits or rolls back the transaction. The connection stays open until you close it.

Here's a check, using a file database so we can watch the connection:

In [ ]:
import sqlite3
from pathlib import Path
import tempfile

db = Path(tempfile.mkdtemp()) / "demo.db"

with sqlite3.connect(db) as conn:
    conn.execute("create table t (x integer)")

try:
    conn.execute("select * from t")
    print("still open after the with block: True")
except sqlite3.ProgrammingError:
    print("still open after the with block: False")

So `with sqlite3.connect(...)` is only half of the job. Wrap it in `closing()` to get both the transaction handling and the close:

```python
with closing(sqlite3.connect(db_path)) as conn:
    ...
```

This is the pattern used in the `src/` scripts.

## 4. Write your own context manager

`@contextmanager` turns a generator function into a context manager. Code before `yield` is setup. Code after `yield` is cleanup, and it runs even if the block fails.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label} took {time.perf_counter() - start:.4f} s")

with timed("sum of squares"):
    total = sum(i * i for i in range(100_000))

print("total:", total)

The `try`/`finally` matters. Without it, a failure inside the block would skip the cleanup line.

Here's the same helper with an error inside the block, to show the cleanup still runs:

In [ ]:
try:
    with timed("failing block"):
        raise ValueError("something went wrong")
except ValueError as err:
    print("caught:", err)

## 5. `suppress`: ignore a specific error

In [ ]:
from contextlib import suppress
from pathlib import Path

with suppress(FileNotFoundError):
    Path("this_file_does_not_exist.txt").unlink()

print("continued past the missing file")

Use `suppress` only for errors you expect and can safely ignore. Otherwise it hides real bugs.

## 6. `redirect_stdout`: capture printed output

`redirect_stdout` sends anything printed inside the block to a different place, such as a string buffer:

In [ ]:
from contextlib import redirect_stdout
import io

buffer = io.StringIO()
with redirect_stdout(buffer):
    print("this text is captured")
    print("so is this")

print("captured:", repr(buffer.getvalue()))

This is handy for testing code that prints, or for saving a report's output to a file.

## Exercises

**Exercise 1.** Write a context manager called `announce` that prints `"start: <label>"` before the block and `"end: <label>"` after it. Make sure `end` prints even if the block raises an error.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
from contextlib import contextmanager

@contextmanager
def announce(label):
    print(f"start: {label}")
    try:
        yield
    finally:
        print(f"end: {label}")

with announce("loading"):
    x = 2 + 2
```

</details>

**Exercise 2.** Use `closing()` to write a function that creates an in-memory SQLite database, creates a table `t` with one row, and returns the row count. The connection must be closed before the function returns.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
from contextlib import closing
import sqlite3

def count_rows():
    with closing(sqlite3.connect(":memory:")) as conn:
        conn.execute("create table t (x integer)")
        conn.execute("insert into t values (1)")
        return conn.execute("select count(*) from t").fetchone()[0]

print(count_rows())
```

</details>

## Summary

- `with` runs setup and cleanup around a block, even when an error happens.
- `closing()` calls `close()` for objects that have one but aren't context managers.
- `with sqlite3.connect(...)` commits or rolls back, but does **not** close. Pair it with `closing()`.
- `@contextmanager` builds your own, using `yield` and `try`/`finally`.
- `suppress` ignores expected errors, and `redirect_stdout` captures printed output.